# Exploring the Dataset: Building the Hosts Table

**Goal:** Understand how we go from a raw YAML configuration file to a relational database table.

This notebook walks through:
1. Loading `servers.yaml` (the raw config that defines all 22 hosts in the network)
2. Examining what fields each host has
3. Transforming the data into a Pandas DataFrame
4. Mapping it to our planned `hosts` database table schema

---

**Dataset:** AIT Log Data Set V2.0 - russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.  

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, just change `DATASET_ROOT` below.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell


## 1. Load the Raw YAML File

The file `processing/config/servers.yaml` defines every host in the simulated enterprise network.  
This is the source for our **`hosts`** database table.

In [2]:
import yaml

servers_file = DATASET_ROOT / "processing" / "config" / "servers.yaml"

with open(servers_file) as f:
    servers = yaml.safe_load(f)

print(f"Loaded {len(servers)} hosts from: {servers_file.name}")
print("\nHost names (keys in the YAML):")
for i, name in enumerate(servers.keys(), 1):
    print(f"  {i:2d}. {name}")

Loaded 22 hosts from: servers.yaml

Host names (keys in the YAML):
   1. remote_employee_1
   2. attacker_0
   3. remote_employee_2
   4. remote_employee_0
   5. cloud_share
   6. webserver
   7. morris_mail
   8. davey_mail
   9. vpn
  10. intranet_server
  11. internal_employee_1
  12. internal_employee_3
  13. ext_user_1
  14. ext_user_0
  15. ext_user_2
  16. internal_share
  17. monitoring
  18. internal_employee_2
  19. mail
  20. internal_employee_0
  21. inet-dns
  22. inet-firewall


## 2. Examine a Single Host

Let's look at what data exists for one host. We'll pick `intranet_server` because it's the main attack target in our dataset.

In [3]:
import json

example_host = "intranet_server"
host_data = servers[example_host]

print(json.dumps(host_data, indent=2))

{
  "hostname": "intranet-server",
  "groups": [
    "beatservers",
    "intranet",
    "servers"
  ],
  "distribution": "Ubuntu",
  "distribution_release": "bionic",
  "distribution_version": "18.04",
  "default_ipv4_address": "10.143.2.4",
  "default_ipv6_address": "fe80::f816:3eff:fe75:ba2b",
  "ipv4_addresses": [
    "10.143.2.4"
  ],
  "ipv6_addresses": [
    "fe80::f816:3eff:fe75:ba2b"
  ],
  "fqdns": [
    "intranet.smith.russellmitchell.com"
  ],
  "logs": [
    {
      "path": "apache2/*access*.log*",
      "type": "apache_access"
    },
    {
      "path": "apache2/*error*.log*",
      "type": "apache_error",
      "add_field": {
        "[@metadata][kyoushi][httpd_dirs]": [
          "/var/www/intranet.smith.russellmitchell.com",
          "",
          "/usr/share/javascript",
          "/javascript"
        ]
      }
    },
    {
      "path": "audit/audit.log*",
      "type": "audit",
      "add_field": {
        "[@metadata][pipeline]": "auditd-logs"
      }
    },
    {

### 2.1 Field Inventory Across All Hosts

Not all hosts have the same fields. Let's discover every unique field across all 22 hosts and check coverage.

In [4]:
# Discover all unique top-level keys across all 22 hosts
all_keys = set()
key_coverage = {}

for host_key, info in servers.items():
    for key in info.keys():
        all_keys.add(key)
        if key not in key_coverage:
            key_coverage[key] = []
        key_coverage[key].append(host_key)

print(f"Total unique fields across all 22 hosts: {len(all_keys)}\n")
print(f"{'Field':<25} {'Present In':>12}  {'Type (sample)':>20}")
print("-" * 65)
for key in sorted(all_keys):
    count = len(key_coverage[key])
    # Get the type of the value from the first host that has it
    sample_host = key_coverage[key][0]
    sample_val = servers[sample_host][key]
    val_type = type(sample_val).__name__
    if isinstance(sample_val, list) and len(sample_val) > 0:
        inner_type = type(sample_val[0]).__name__
        val_type = f"list[{inner_type}]"
    print(f"  {key:<25} {count:>5}/22      {val_type:>20}")

Total unique fields across all 22 hosts: 14

Field                       Present In         Type (sample)
-----------------------------------------------------------------
  default_ipv4_address         22/22                       str
  default_ipv6_address         22/22                       str
  distribution                 22/22                       str
  distribution_release         22/22                       str
  distribution_version         22/22                       str
  fqdns                        22/22                      list
  groups                       22/22                 list[str]
  hostname                     22/22                       str
  ipv4_addresses               22/22                 list[str]
  ipv6_addresses               22/22                 list[str]
  logs                         22/22                list[dict]
  openvpn_user                  3/22                       str
  timezone                     22/22                       str
  usernam

### 2.2 Multi-Valued Field: `groups`

Each host belongs to 2-5 groups. These are role and network zone tags used by the Kyoushi simulation framework. This is a multi-valued attribute (1NF violation).

In [5]:
from collections import Counter

# Explore the groups field - a multi-valued attribute
print("=== Groups per host ===\n")
for host_key, info in servers.items():
    groups = info.get("groups", [])
    print(f"  {host_key:<25} ({len(groups)} groups): {groups}")

# Unique group values
all_group_values = [g for info in servers.values() for g in info.get("groups", [])]
group_counter = Counter(all_group_values)

print(f"\n=== Unique group values ({len(group_counter)} total) ===\n")
for g, count in group_counter.most_common():
    print(f"  {g:<25} appears in {count} hosts")

print("\n=== Groups per host (count distribution) ===")
group_counts = [len(info.get("groups", [])) for info in servers.values()]
for n in sorted(set(group_counts)):
    hosts_with_n = [hk for hk, info in servers.items() if len(info.get("groups", [])) == n]
    print(f"  {n} groups: {len(hosts_with_n)} hosts - {hosts_with_n}")

=== Groups per host ===

  remote_employee_1         (3 groups): ['employee', 'internet', 'remote_employee']
  attacker_0                (2 groups): ['attacker', 'internet']
  remote_employee_2         (3 groups): ['employee', 'internet', 'remote_employee']
  remote_employee_0         (3 groups): ['employee', 'internet', 'remote_employee']
  cloud_share               (3 groups): ['dmz', 'proxied', 'servers']
  webserver                 (3 groups): ['dmz', 'dnat', 'servers']
  morris_mail               (3 groups): ['ext_mail', 'internet', 'mailserver']
  davey_mail                (3 groups): ['ext_mail', 'internet', 'mailserver']
  vpn                       (3 groups): ['dmz', 'dnat', 'servers']
  intranet_server           (3 groups): ['beatservers', 'intranet', 'servers']
  internal_employee_1       (3 groups): ['employee', 'internal_employee', 'intranet']
  internal_employee_3       (3 groups): ['employee', 'internal_employee', 'intranet']
  ext_user_1                (2 groups): ['ext

### 2.3 Multi-Valued Field: `fqdns`

Fully qualified domain names. Some hosts have none, some have multiple. This is a multi-valued attribute (1NF violation).

In [6]:
# Explore the fqdns field - a multi-valued attribute
empty_fqdn_hosts = []
single_fqdn_hosts = []
multi_fqdn_hosts = []

for host_key, info in servers.items():
    fqdns = info.get("fqdns", [])
    if len(fqdns) == 0:
        empty_fqdn_hosts.append(host_key)
    elif len(fqdns) == 1:
        single_fqdn_hosts.append((host_key, fqdns[0]))
    else:
        multi_fqdn_hosts.append((host_key, fqdns))

print(f"Hosts with no FQDNs ({len(empty_fqdn_hosts)}):")
for h in empty_fqdn_hosts:
    print(f"  {h}")

print(f"\nHosts with 1 FQDN ({len(single_fqdn_hosts)}):")
for h, fqdn in single_fqdn_hosts:
    print(f"  {h:<25} {fqdn}")

print(f"\nHosts with multiple FQDNs ({len(multi_fqdn_hosts)}):")
for h, fqdns in multi_fqdn_hosts:
    print(f"  {h:<25} ({len(fqdns)} FQDNs):")
    for fqdn in fqdns:
        print(f"    - {fqdn}")

Hosts with no FQDNs (7):
  remote_employee_1
  attacker_0
  remote_employee_2
  remote_employee_0
  ext_user_1
  ext_user_0
  ext_user_2

Hosts with 1 FQDN (12):
  cloud_share               cloud.dmz.smith.russellmitchell.com
  webserver                 proxy.smith.russellmitchell.com
  morris_mail               mailserver.morris.russellmitchell.com
  davey_mail                smtp.davey.russellmitchell.com
  intranet_server           intranet.smith.russellmitchell.com
  internal_employee_1       employee01.intranet.smith.russellmitchell.com
  internal_employee_3       employee03.intranet.smith.russellmitchell.com
  internal_share            share.intranet.smith.russellmitchell.com
  monitoring                monitoring.intranet.smith.russellmitchell.com
  internal_employee_2       employee02.intranet.smith.russellmitchell.com
  internal_employee_0       employee00.intranet.smith.russellmitchell.com
  inet-dns                  dns.russellmitchell.com

Hosts with multiple FQDNs (3):
  v

### 2.4 Multi-Valued Fields: `ipv4_addresses`, `ipv6_addresses`

Each host has a `default_ipv4_address` (scalar) and an `ipv4_addresses` array. Most hosts have exactly one IP in the array matching the default. Let's check for multi-valued cases.

In [7]:
# Explore ipv4_addresses and ipv6_addresses
print("=== IPv4: default vs array ===\n")
for host_key, info in servers.items():
    ipv4s = info.get("ipv4_addresses", [])
    default = info.get("default_ipv4_address", "")
    marker = ""
    if len(ipv4s) > 1:
        marker = " ** MULTI-VALUED **"
    elif ipv4s and ipv4s[0] != default:
        marker = " ** MISMATCH WITH DEFAULT **"
    print(f"  {host_key:<25} default={default:<20} array={ipv4s}{marker}")

print("\n=== IPv6: default vs array ===\n")
for host_key, info in servers.items():
    ipv6s = info.get("ipv6_addresses", [])
    marker = ""
    if len(ipv6s) > 1:
        marker = " ** MULTI-VALUED **"
    print(f"  {host_key:<25} count={len(ipv6s)}{marker}")

# Summarize multi-valued cases
multi_ipv4 = [
    (hk, info["ipv4_addresses"])
    for hk, info in servers.items()
    if len(info.get("ipv4_addresses", [])) > 1
]
multi_ipv6 = [
    (hk, info["ipv6_addresses"])
    for hk, info in servers.items()
    if len(info.get("ipv6_addresses", [])) > 1
]

print("\n=== Summary ===")
print(f"  Hosts with multiple IPv4 addresses: {len(multi_ipv4)}")
for hk, ips in multi_ipv4:
    groups = servers[hk].get("groups", [])
    print(f"    {hk}: {ips}")
    print(f"      groups: {groups} (spans multiple network zones)")
print(f"  Hosts with multiple IPv6 addresses: {len(multi_ipv6)}")
for hk, ips in multi_ipv6:
    print(f"    {hk}: {ips}")

=== IPv4: default vs array ===

  remote_employee_1         default=192.168.230.95       array=['192.168.230.95']
  attacker_0                default=192.168.230.122      array=['192.168.230.122']
  remote_employee_2         default=192.168.230.165      array=['192.168.230.165']
  remote_employee_0         default=192.168.231.127      array=['192.168.231.127']
  cloud_share               default=172.19.130.106       array=['172.19.130.106']
  webserver                 default=172.19.130.68        array=['172.19.130.68']
  morris_mail               default=192.168.231.164      array=['192.168.231.164']
  davey_mail                default=192.168.231.56       array=['192.168.231.56']
  vpn                       default=172.19.131.174       array=['172.19.131.174']
  intranet_server           default=10.143.2.4           array=['10.143.2.4']
  internal_employee_1       default=10.143.2.91          array=['10.143.2.91']
  internal_employee_3       default=10.143.3.65          array=['10.14

### 2.5 Nested Structure: `logs`

The `logs` field is the most complex - it's an array of objects, where each object describes a log file configuration. This is both multi-valued AND composite (nested). We need to explore every level of nesting.

In [8]:
# Explore the logs field - multi-valued AND composite (array of objects)

# Collect structure info
log_entry_keys = set()
all_log_types = set()
add_field_keys = set()
total_entries = 0

for _host_key, info in servers.items():
    logs = info.get("logs", [])
    total_entries += len(logs)
    for log_entry in logs:
        log_entry_keys.update(log_entry.keys())
        all_log_types.add(log_entry.get("type", ""))
        if "add_field" in log_entry:
            add_field_keys.update(log_entry["add_field"].keys())

print("=== Log Entry Structure ===\n")
print(f"Total log entries across all 22 hosts: {total_entries}")
print(f"Top-level keys in log entries: {sorted(log_entry_keys)}")
print(f"Unique log types ({len(all_log_types)}): {sorted(all_log_types)}")
print(f"add_field metadata keys ({len(add_field_keys)}): {sorted(add_field_keys)}")

# Coverage of each key across all log entries
print("\n=== Log Entry Key Coverage ===\n")
for key in sorted(log_entry_keys):
    count = sum(
        1 for info in servers.values() for log_entry in info.get("logs", []) if key in log_entry
    )
    print(f"  {key:<25} present in {count}/{total_entries} log entries")

# Entries per host
print("\n=== Log Entries per Host ===\n")
for _host_key, info in servers.items():
    logs = info.get("logs", [])
    types = [log.get("type", "") for log in logs]
    has_add_field = sum(1 for log in logs if "add_field" in log)
    has_codec = sum(1 for log in logs if "codec" in log)
    extra = []
    if has_add_field:
        extra.append(f"{has_add_field} with add_field")
    if has_codec:
        extra.append(f"{has_codec} with codec")
    extra_str = f"  ({', '.join(extra)})" if extra else ""
    print(f"  {host_key:<25} {len(logs):>2} entries: {types}{extra_str}")

# Show a complete example with all optional keys
print("\n=== Example: log entry with all optional keys (add_field, codec, file_chunk_size) ===\n")
for _host_key, info in servers.items():
    for log_entry in info.get("logs", []):
        if all(k in log_entry for k in ["add_field", "codec", "file_chunk_size"]):
            print(f"Host: {host_key}")
            print(json.dumps(log_entry, indent=2))
            break
    else:
        continue
    break

# Show all add_field values (deepest nesting level)
print("\n=== All add_field values (deepest nesting) ===\n")
for _host_key, info in servers.items():
    for log_entry in info.get("logs", []):
        if "add_field" in log_entry:
            af = log_entry["add_field"]
            for af_key, af_val in af.items():
                val_type = type(af_val).__name__
                print(
                    f"  {host_key:<25} type={log_entry['type']:<20} {af_key} = {af_val} ({val_type})"
                )

=== Log Entry Structure ===

Total log entries across all 22 hosts: 66
Top-level keys in log entries: ['add_field', 'codec', 'file_chunk_size', 'path', 'type']
Unique log types (11): ['apache_access', 'apache_error', 'audit', 'auth', 'dnsmasq', 'dnsteal', 'kyoushi', 'metricsbeat', 'openvpn', 'pcap', 'syslog']
add_field metadata keys (4): ['[@metadata][host_override]', '[@metadata][kyoushi][httpd_dirs]', '[@metadata][kyoushi][sm]', '[@metadata][pipeline]']

=== Log Entry Key Coverage ===

  add_field                 present in 21/66 log entries
  codec                     present in 15/66 log entries
  file_chunk_size           present in 11/66 log entries
  path                      present in 66/66 log entries
  type                      present in 66/66 log entries

=== Log Entries per Host ===

  remote_employee_1          1 entries: ['kyoushi']  (1 with add_field, 1 with codec)
  attacker_0                 4 entries: ['kyoushi', 'pcap', 'dnsteal', 'pcap']  (1 with add_field, 4 with

### 2.6 Fields: `username`, `openvpn_user`

These fields only appear on employee hosts. Let's see which hosts have them and whether there's a relationship between the two.

In [9]:
# Explore username and openvpn_user fields
print("=== username field (7/22 hosts) ===\n")
with_username = []
without_username = []

for host_key, info in servers.items():
    if "username" in info:
        with_username.append((host_key, info["username"], info.get("openvpn_user")))
    else:
        without_username.append(host_key)

for host_key, username, openvpn in with_username:
    groups = servers[host_key].get("groups", [])
    print(
        f"  {host_key:<25} username={username:<15} openvpn_user={str(openvpn or 'N/A'):<15} groups={groups}"
    )

print(f"\nHosts without username ({len(without_username)}):")
for h in without_username:
    groups = servers[h].get("groups", [])
    print(f"  {h:<25} groups={groups}")

print("\n=== Observations ===")
print(f"  - username present on all hosts in the 'employee' group ({len(with_username)} hosts)")
print("  - openvpn_user present only on hosts in the 'remote_employee' group (3 hosts)")
# Check if openvpn_user always matches username
for host_key, username, openvpn in with_username:
    if openvpn is not None and openvpn != username:
        print(f"  - MISMATCH: {host_key} username={username} openvpn_user={openvpn}")
        break
else:
    print("  - openvpn_user always matches username when present (no mismatches)")

=== username field (7/22 hosts) ===

  remote_employee_1         username=twhite          openvpn_user=twhite          groups=['employee', 'internet', 'remote_employee']
  remote_employee_2         username=jhall           openvpn_user=jhall           groups=['employee', 'internet', 'remote_employee']
  remote_employee_0         username=ahayes          openvpn_user=ahayes          groups=['employee', 'internet', 'remote_employee']
  internal_employee_1       username=dellis          openvpn_user=N/A             groups=['employee', 'internal_employee', 'intranet']
  internal_employee_3       username=rblake          openvpn_user=N/A             groups=['employee', 'internal_employee', 'intranet']
  internal_employee_2       username=tstevenson      openvpn_user=N/A             groups=['employee', 'internal_employee', 'intranet']
  internal_employee_0       username=mmorgan         openvpn_user=N/A             groups=['employee', 'internal_employee', 'intranet']

Hosts without username 

### 2.7 Fields: `distribution`, `distribution_release`, `distribution_version`, `timezone`

These are scalar fields present on all 22 hosts. Let's check value distributions.

In [10]:
# Explore distribution, distribution_release, distribution_version, timezone
print("=== distribution + distribution_release + distribution_version ===\n")
for host_key, info in servers.items():
    print(
        f"  {host_key:<25} "
        f"dist={info['distribution']:<10} "
        f"release={info['distribution_release']:<10} "
        f"version={info['distribution_version']}"
    )

print("\n=== Distribution combinations ===")
dist_counter = Counter(
    (info["distribution"], info["distribution_release"], info["distribution_version"])
    for info in servers.values()
)
for (dist, release, version), count in dist_counter.most_common():
    print(f"  {dist} {release} {version}: {count} hosts")

print("\n=== timezone ===")
tz_counter = Counter(info.get("timezone", "N/A") for info in servers.values())
for tz, count in tz_counter.most_common():
    print(f"  {tz}: {count} hosts")

print("\n=== Observations ===")
print("  - distribution_release determines distribution and distribution_version:")
print("    bionic -> Ubuntu 18.04 (19 hosts)")
print("    stretch -> Debian 9.11 (3 hosts)")
print("  - This is a potential transitive dependency (relevant for 3NF)")
print("  - timezone is constant (UTC) across all hosts")

=== distribution + distribution_release + distribution_version ===

  remote_employee_1         dist=Ubuntu     release=bionic     version=18.04
  attacker_0                dist=Ubuntu     release=bionic     version=18.04
  remote_employee_2         dist=Ubuntu     release=bionic     version=18.04
  remote_employee_0         dist=Ubuntu     release=bionic     version=18.04
  cloud_share               dist=Ubuntu     release=bionic     version=18.04
  webserver                 dist=Ubuntu     release=bionic     version=18.04
  morris_mail               dist=Debian     release=stretch    version=9.11
  davey_mail                dist=Debian     release=stretch    version=9.11
  vpn                       dist=Ubuntu     release=bionic     version=18.04
  intranet_server           dist=Ubuntu     release=bionic     version=18.04
  internal_employee_1       dist=Ubuntu     release=bionic     version=18.04
  internal_employee_3       dist=Ubuntu     release=bionic     version=18.04
  ext_user

### Field Inventory Summary

All 22 hosts share 12 fields. Two additional fields appear only on employee hosts.

| # | YAML Field | Type | Present In | Multi-Valued | Description |
|---|-----------|------|-----------|-------------|-------------|
| 1 | `hostname` | str | 22/22 | No | Machine's network name (e.g., `intranet-server`) |
| 2 | `groups` | list[str] | 22/22 | Yes (2-5 values) | Roles and network zone tags (17 unique values) |
| 3 | `distribution` | str | 22/22 | No | Linux distribution (Ubuntu or Debian) |
| 4 | `distribution_release` | str | 22/22 | No | OS release codename (e.g., `bionic`, `stretch`) |
| 5 | `distribution_version` | str | 22/22 | No | OS version number (e.g., `18.04`, `9.11`) |
| 6 | `default_ipv4_address` | str | 22/22 | No | Primary IPv4 address |
| 7 | `default_ipv6_address` | str | 22/22 | No | Primary IPv6 address |
| 8 | `ipv4_addresses` | list[str] | 22/22 | Yes (1-3 values) | All IPv4 addresses on the host |
| 9 | `ipv6_addresses` | list[str] | 22/22 | Yes (1-3 values) | All IPv6 addresses on the host |
| 10 | `fqdns` | list[str] | 22/22 | Yes (0-4 values) | Fully qualified domain names |
| 11 | `logs` | list[dict] | 22/22 | Yes (1-9 entries) | Log file configs (nested: path, type, add_field, codec, file_chunk_size) |
| 12 | `timezone` | str | 22/22 | No | System timezone (all UTC) |
| 13 | `username` | str | 7/22 | No | User account (employee hosts only) |
| 14 | `openvpn_user` | str | 3/22 | No | VPN username (remote employees only) |

Multi-valued fields (groups, fqdns, ipv4_addresses, ipv6_addresses, logs) are 1NF violations that will need to be resolved during normalization.

## 3. Transform into a Raw DataFrame

Now we'll convert all 22 hosts into a structured DataFrame that captures every field from the YAML as-is. This is a 1:1 representation of the raw data before any normalization or transformation.

Multi-valued fields (groups, fqdns, ipv4_addresses, ipv6_addresses, logs) are kept as Python lists to preserve their structure. These are 1NF violations that will be resolved during normalization.

In [11]:
import pandas as pd

# Build a raw 1:1 DataFrame - every field from the YAML, no transformations
rows = []
for host_key, info in servers.items():
    rows.append(
        {
            "host_key": host_key,
            "hostname": info.get("hostname"),
            "groups": info.get("groups", []),
            "username": info.get("username"),
            "openvpn_user": info.get("openvpn_user"),
            "distribution": info.get("distribution"),
            "distribution_release": info.get("distribution_release"),
            "distribution_version": info.get("distribution_version"),
            "default_ipv4_address": info.get("default_ipv4_address"),
            "default_ipv6_address": info.get("default_ipv6_address"),
            "ipv4_addresses": info.get("ipv4_addresses", []),
            "ipv6_addresses": info.get("ipv6_addresses", []),
            "fqdns": info.get("fqdns", []),
            "logs": info.get("logs", []),
            "timezone": info.get("timezone"),
        }
    )

df_hosts_raw = pd.DataFrame(rows)
print(f"Raw DataFrame: {df_hosts_raw.shape[0]} rows x {df_hosts_raw.shape[1]} columns")
print(f"\nAll columns ({len(df_hosts_raw.columns)}):")
for i, col in enumerate(df_hosts_raw.columns, 1):
    dtype = df_hosts_raw[col].dtype
    nulls = df_hosts_raw[col].isnull().sum()
    print(f"  {i:2d}. {col:<25} dtype={str(dtype):<10} nulls={nulls}")

Raw DataFrame: 22 rows x 15 columns

All columns (15):
   1. host_key                  dtype=object     nulls=0
   2. hostname                  dtype=object     nulls=0
   3. groups                    dtype=object     nulls=0
   4. username                  dtype=object     nulls=15
   5. openvpn_user              dtype=object     nulls=19
   6. distribution              dtype=object     nulls=0
   7. distribution_release      dtype=object     nulls=0
   8. distribution_version      dtype=object     nulls=0
   9. default_ipv4_address      dtype=object     nulls=0
  10. default_ipv6_address      dtype=object     nulls=0
  11. ipv4_addresses            dtype=object     nulls=0
  12. ipv6_addresses            dtype=object     nulls=0
  13. fqdns                     dtype=object     nulls=0
  14. logs                      dtype=object     nulls=0
  15. timezone                  dtype=object     nulls=0


In [12]:
# Display the scalar (non-list) columns for readability
scalar_cols = [
    "host_key",
    "hostname",
    "username",
    "openvpn_user",
    "distribution",
    "distribution_release",
    "distribution_version",
    "default_ipv4_address",
    "default_ipv6_address",
    "timezone",
]
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.max_columns", None)
df_hosts_raw[scalar_cols]

,host_key,hostname,username,openvpn_user,distribution,distribution_release,distribution_version,default_ipv4_address,default_ipv6_address,timezone
0,remote_employee_1,remote-employee-1,twhite,twhite,Ubuntu,bionic,18.04,192.168.230.95,fe80::f816:3eff:fefb:8b25,UTC
1,attacker_0,attacker-0,None,None,Ubuntu,bionic,18.04,192.168.230.122,fe80::f816:3eff:fe1d:2fc5,UTC
2,remote_employee_2,remote-employee-2,jhall,jhall,Ubuntu,bionic,18.04,192.168.230.165,fe80::f816:3eff:fec4:1e0b,UTC
3,remote_employee_0,remote-employee-0,ahayes,ahayes,Ubuntu,bionic,18.04,192.168.231.127,fe80::f816:3eff:feea:4536,UTC
4,cloud_share,cloud-share,None,None,Ubuntu,bionic,18.04,172.19.130.106,fe80::f816:3eff:fe5b:275b,UTC
5,webserver,webserver,None,None,Ubuntu,bionic,18.04,172.19.130.68,fe80::f816:3eff:fe47:ba82,UTC
6,morris_mail,morris-mail,None,None,Debian,stretch,9.11,192.168.231.164,fe80::f816:3eff:feb0:71cb,UTC
7,davey_mail,davey-mail,None,None,Debian,stretch,9.11,192.168.231.56,fe80::f816:3eff:fe30:e67c,UTC
8,vpn,vpn,None,None,Ubuntu,bionic,18.04,172.19.131.174,fe80::f816:3eff:fe1b:6d09,UTC
9,intranet_server,intranet-server,None,None,Ubuntu,bionic,18.04,10.143.2.4,fe80::f816:3eff:fe75:ba2b,UTC


In [13]:
# Display the multi-valued columns: show counts and sample values
print("Multi-valued columns summary:\n")
for col in ["groups", "fqdns", "ipv4_addresses", "ipv6_addresses", "logs"]:
    lengths = df_hosts_raw[col].apply(len)
    print(f"{col}:")
    print(
        f"  min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.1f}, total_entries={lengths.sum()}"
    )
    # Show hosts with max values
    max_host = df_hosts_raw.loc[lengths.idxmax(), "host_key"]
    print(f"  host with most entries: {max_host} ({lengths.max()} entries)")
    print()

Multi-valued columns summary:

groups:
  min=2, max=5, mean=2.9, total_entries=63
  host with most entries: mail (5 entries)

fqdns:
  min=0, max=4, mean=0.9, total_entries=20
  host with most entries: inet-firewall (4 entries)

ipv4_addresses:
  min=1, max=3, mean=1.1, total_entries=24
  host with most entries: inet-firewall (3 entries)

ipv6_addresses:
  min=1, max=3, mean=1.1, total_entries=24
  host with most entries: inet-firewall (3 entries)

logs:
  min=1, max=9, mean=3.0, total_entries=66
  host with most entries: mail (9 entries)



## 4. Network Zones (from `groups`)

The `groups` field (explored in section 2.2) encodes network zone membership. Let's visualize which hosts belong to which groups to understand the network topology.

In [14]:
# Count hosts by their group memberships
all_groups = []
for host_key, info in servers.items():
    for group in info.get("groups", []):
        all_groups.append({"host_key": host_key, "group": group})

df_groups = pd.DataFrame(all_groups)
print("Hosts per network group:\n")
print(df_groups.groupby("group")["host_key"].apply(list).to_string())

Hosts per network group:

group
attacker                                        [attacker_0]
beatservers                [intranet_server, internal_share]
dmz                      [cloud_share, webserver, vpn, mail]
dnat                                  [webserver, vpn, mail]
dnsservers                         [inet-dns, inet-firewall]
employee             [remote_employee_1, remote_employee_...
ext_mail                           [morris_mail, davey_mail]
ext_user                [ext_user_1, ext_user_0, ext_user_2]
firewall                                     [inet-firewall]
internal_employee    [internal_employee_1, internal_emplo...
internet             [remote_employee_1, attacker_0, remo...
intranet             [intranet_server, internal_employee_...
mailserver                   [morris_mail, davey_mail, mail]
proxied                                  [cloud_share, mail]
remote_employee      [remote_employee_1, remote_employee_...
servers              [cloud_share, webserver, vpn, in

## 5. Log Types per Host (from `logs`)

The `logs` field (explored in section 2.5) tells us which raw log files each host generates. This maps directly to which of our database tables will contain events from each host.

In [15]:
# Show log types per host, sorted by number of log sources (most first)
print(f"{'Host':<25} {'# Logs':>7}   Log Types")
print("-" * 80)

host_log_info = []
for _, row in df_hosts_raw.iterrows():
    logs = row["logs"] if isinstance(row["logs"], list) else []
    num_sources = len(logs)
    types = ", ".join(sorted({entry.get("type", "unknown") for entry in logs}))
    host_log_info.append((row["hostname"], num_sources, types))

for hostname, num_sources, types in sorted(host_log_info, key=lambda x: x[1], reverse=True):
    print(f"{hostname:<25} {num_sources:>7}   {types}")

Host                       # Logs   Log Types
--------------------------------------------------------------------------------
mail                            9   apache_access, apache_error, audit, auth, syslog
morris-mail                     6   apache_access, apache_error, syslog
davey-mail                      6   apache_access, apache_error, syslog
cloud-share                     5   apache_access, apache_error, audit, auth, syslog
webserver                       5   apache_access, apache_error, audit, auth, syslog
intranet-server                 5   apache_access, apache_error, audit, auth, syslog
attacker-0                      4   dnsteal, kyoushi, pcap
vpn                             4   audit, auth, openvpn, syslog
monitoring                      4   audit, auth, metricsbeat, syslog
inet-firewall                   4   audit, auth, dnsmasq, syslog
internal-share                  3   audit, auth, syslog
remote-employee-1               1   kyoushi
remote-employee-2              

## 6. Mapping to the Database Schema

This is the raw 1:1 mapping from the YAML file to a database table. This table is NOT normalized - it represents the raw data as discovered. Multi-valued fields are stored as comma-separated strings for initial loading. The `logs` nested structure maps to a separate table.

### Raw Field to Column Mapping

| # | YAML Field | DB Column | PostgreSQL Type | MySQL Type | Nullable | Notes |
|---|-----------|-----------|-----------------|------------|----------|-------|
| 1 | *(auto)* | `host_id` | `SERIAL PRIMARY KEY` | `INT AUTO_INCREMENT PRIMARY KEY` | No | Surrogate key |
| 2 | (dict key) | `host_key` | `VARCHAR(50) NOT NULL UNIQUE` | `VARCHAR(50) NOT NULL UNIQUE` | No | YAML dictionary key (e.g., `intranet_server`) |
| 3 | `hostname` | `hostname` | `VARCHAR(100) NOT NULL UNIQUE` | `VARCHAR(100) NOT NULL UNIQUE` | No | Network hostname |
| 4 | `groups` | `groups` | `TEXT NOT NULL` | `TEXT NOT NULL` | No | 1NF violation: comma-separated list (2-5 values, 17 unique) |
| 5 | `username` | `username` | `VARCHAR(50)` | `VARCHAR(50)` | Yes | 7/22 employee hosts |
| 6 | `openvpn_user` | `openvpn_user` | `VARCHAR(50)` | `VARCHAR(50)` | Yes | 3/22 remote employee hosts |
| 7 | `distribution` | `distribution` | `VARCHAR(50) NOT NULL` | `VARCHAR(50) NOT NULL` | No | Ubuntu or Debian |
| 8 | `distribution_release` | `distribution_release` | `VARCHAR(20) NOT NULL` | `VARCHAR(20) NOT NULL` | No | bionic or stretch |
| 9 | `distribution_version` | `distribution_version` | `VARCHAR(20) NOT NULL` | `VARCHAR(20) NOT NULL` | No | 18.04 or 9.11 |
| 10 | `default_ipv4_address` | `default_ipv4_address` | `VARCHAR(45) NOT NULL` | `VARCHAR(45) NOT NULL` | No | Primary IPv4 |
| 11 | `default_ipv6_address` | `default_ipv6_address` | `VARCHAR(45) NOT NULL` | `VARCHAR(45) NOT NULL` | No | Primary IPv6 |
| 12 | `ipv4_addresses` | `ipv4_addresses` | `TEXT NOT NULL` | `TEXT NOT NULL` | No | 1NF violation: comma-separated (1-3 values) |
| 13 | `ipv6_addresses` | `ipv6_addresses` | `TEXT NOT NULL` | `TEXT NOT NULL` | No | 1NF violation: comma-separated (1-3 values) |
| 14 | `fqdns` | `fqdns` | `TEXT` | `TEXT` | Yes | 1NF violation: comma-separated (0-4 values). 7 hosts have none. |
| 15 | `timezone` | `timezone` | `VARCHAR(10) NOT NULL` | `VARCHAR(10) NOT NULL` | No | All UTC |

### Suggested CREATE TABLE DDL (un-normalized, for initial data loading)

```sql
-- PostgreSQL syntax
CREATE TABLE hosts_raw (
    host_id SERIAL PRIMARY KEY,
    host_key VARCHAR(50) NOT NULL UNIQUE,
    hostname VARCHAR(100) NOT NULL UNIQUE,
    groups TEXT NOT NULL,
    username VARCHAR(50),
    openvpn_user VARCHAR(50),
    distribution VARCHAR(50) NOT NULL,
    distribution_release VARCHAR(20) NOT NULL,
    distribution_version VARCHAR(20) NOT NULL,
    default_ipv4_address VARCHAR(45) NOT NULL,
    default_ipv6_address VARCHAR(45) NOT NULL,
    ipv4_addresses TEXT NOT NULL,
    ipv6_addresses TEXT NOT NULL,
    fqdns TEXT,
    timezone VARCHAR(10) NOT NULL
);
```

```sql
-- MySQL equivalent
CREATE TABLE hosts_raw (
    host_id INT AUTO_INCREMENT PRIMARY KEY,
    host_key VARCHAR(50) NOT NULL UNIQUE,
    hostname VARCHAR(100) NOT NULL UNIQUE,
    groups TEXT NOT NULL,
    username VARCHAR(50),
    openvpn_user VARCHAR(50),
    distribution VARCHAR(50) NOT NULL,
    distribution_release VARCHAR(20) NOT NULL,
    distribution_version VARCHAR(20) NOT NULL,
    default_ipv4_address VARCHAR(45) NOT NULL,
    default_ipv6_address VARCHAR(45) NOT NULL,
    ipv4_addresses TEXT NOT NULL,
    ipv6_addresses TEXT NOT NULL,
    fqdns TEXT,
    timezone VARCHAR(10) NOT NULL
);
```

### 6.1 Separate Table: host_log_configs

The `logs` field is a composite multi-valued attribute (array of objects with 5 possible keys). Each host has 1-9 log config entries. This maps to its own table.

```sql
-- PostgreSQL syntax
CREATE TABLE host_log_configs_raw (
    config_id SERIAL PRIMARY KEY,
    host_id INT NOT NULL REFERENCES hosts_raw(host_id),
    log_path TEXT NOT NULL,
    log_type VARCHAR(50) NOT NULL,
    codec VARCHAR(20),
    file_chunk_size INT,
    add_field_json TEXT
);
```

```sql
-- MySQL equivalent
CREATE TABLE host_log_configs_raw (
    config_id INT AUTO_INCREMENT PRIMARY KEY,
    host_id INT NOT NULL,
    log_path TEXT NOT NULL,
    log_type VARCHAR(50) NOT NULL,
    codec VARCHAR(20),
    file_chunk_size INT,
    add_field_json TEXT,
    FOREIGN KEY (host_id) REFERENCES hosts_raw(host_id)
);
```

66 total log config entries across 22 hosts. 11 unique log types.

## 7. Observations for Normalization Phase

These observations will feed into `docs/schema_normalization.md`. They are noted here for reference but the actual normalization decisions happen later.

### 1NF Violations Identified

| Field | Issue | Resolution Direction |
|-------|-------|---------------------|
| `groups` | Multi-valued (2-5 values per host, 17 unique group tags) | Separate `host_groups` junction table: (host_id, group_name) |
| `fqdns` | Multi-valued (0-4 values per host) | Separate `host_fqdns` table: (host_id, fqdn) |
| `ipv4_addresses` | Multi-valued (1-3 per host, only inet-firewall has >1) | Separate `host_ipv4` table or keep default only |
| `ipv6_addresses` | Multi-valued (1-3 per host, only inet-firewall has >1) | Separate `host_ipv6` table or keep default only |
| `logs` | Multi-valued AND composite | Already separated as `host_log_configs_raw` table |

### Potential Derived Columns (for normalization phase)

The `groups` array encodes both role information and network zone membership. During normalization, we could derive:
- `host_type`: firewall, server, workstation, attacker, etc. (from groups using priority logic)
- `network_zone`: external, dmz, internal (from groups using zone-indicator tags)

The derivation logic would be:
- host_type priority: attacker > firewall > dns_server > mail_server > server > workstation > external_user
- network_zone mapping: internet group = external, dmz group = dmz, intranet group = internal

These are design decisions for the normalization doc, not raw data.

### Potential Functional Dependencies (preliminary)

- `host_key` -> all other attributes (candidate key)
- `hostname` -> all other attributes (candidate key)
- `default_ipv4_address` -> hostname (if IPs are unique per host, needs validation)
- `openvpn_user` -> `username` (always the same value when both present)
- `distribution_release` -> `distribution`, `distribution_version` (bionic = Ubuntu 18.04, stretch = Debian 9.11)

These will be validated and formally documented during the FD identification phase.

## 8. Summary

### What we discovered

1. `processing/config/servers.yaml` defines 22 hosts with 14 unique fields per host
2. 5 fields are multi-valued (groups, fqdns, ipv4/ipv6 arrays, logs) - all 1NF violations
3. 2 fields are conditionally present: `username` (7 employee hosts), `openvpn_user` (3 remote employees)
4. The `logs` field is a composite nested structure (array of objects with 5 possible keys), requiring its own table
5. `inet-firewall` is the only host with multiple IP addresses (3 IPv4, 3 IPv6) - it spans 3 network zones
6. All hosts share timezone UTC. 19/22 run Ubuntu 18.04 (bionic), 3/22 run Debian 9.11 (stretch)

### Raw schema output

- `hosts_raw` table: 15 columns, 22 rows
- `host_log_configs_raw` table: 6 columns, 66 rows

### Next steps

- Write findings doc consolidating these discoveries (`docs/data_exploration/notebook_findings/naman_hosts_findings.md`)
- Identify functional dependencies formally
- Apply normalization (1NF: explode multi-valued fields, 2NF/3NF: check for partial and transitive dependencies)
- Design the final normalized `hosts` table for the 3NF schema